# Tratamento dos dados .ZIP e estruturação dos dados da B3

In [2]:
import zipfile
from pathlib import Path

import pandas as pd

In [3]:
PASTA_PROJETO = Path.cwd().parent
PASTA_DADOS = PASTA_PROJETO / "dados"
PASTA_DADOS_TRATADOS = PASTA_PROJETO / "dados_tratados"

print("Pasta do projeto:", PASTA_PROJETO)
print("Pasta dos dados brutos:", PASTA_DADOS)
print("Pasta dos dados tratados:", PASTA_DADOS_TRATADOS)

Pasta do projeto: c:\Users\jayni\OneDrive\Documentos\pairs_trading_b3
Pasta dos dados brutos: c:\Users\jayni\OneDrive\Documentos\pairs_trading_b3\dados
Pasta dos dados tratados: c:\Users\jayni\OneDrive\Documentos\pairs_trading_b3\dados_tratados


### 1. Verificação dos arquivos brutos da B3

Nesta etapa, verificamos se os arquivos históricos da B3 foram baixados corretamente e se o Python consegue acessar.

In [4]:
arquivos_zip = sorted(PASTA_DADOS.glob("COTAHIST_A*.ZIP"))

print(f"Quantidade de arquivos encontrados: {len(arquivos_zip)}")

for arquivo in arquivos_zip:
    print(arquivo.name)

Quantidade de arquivos encontrados: 16
COTAHIST_A2010.ZIP
COTAHIST_A2011.ZIP
COTAHIST_A2012.ZIP
COTAHIST_A2013.ZIP
COTAHIST_A2014.ZIP
COTAHIST_A2015.ZIP
COTAHIST_A2016.ZIP
COTAHIST_A2017.ZIP
COTAHIST_A2018.ZIP
COTAHIST_A2019.ZIP
COTAHIST_A2020.ZIP
COTAHIST_A2021.ZIP
COTAHIST_A2022.ZIP
COTAHIST_A2023.ZIP
COTAHIST_A2024.ZIP
COTAHIST_A2025.ZIP


### 2. Abertura do arquivo ZIP de 2010 da COTAHIST para teste

Nesta etapa, abrimos o arquivo COTAHIST de 2010 para verificar qual arquivo TXT existe dentro do ZIP. Esse teste é importante antes de processar todos os anos, pois confirma que o Python consegue acessar corretamente a estrutura interna do arquivo baixado da B3.

In [5]:
import zipfile

arquivo_teste = PASTA_DADOS / "COTAHIST_A2010.ZIP"

with zipfile.ZipFile(arquivo_teste, "r") as zip_ref:
    arquivos_dentro_zip = zip_ref.namelist()

arquivos_dentro_zip

['COTAHIST_A2010.TXT']

In [6]:
with zipfile.ZipFile(arquivo_teste, "r") as zip_ref:
    nome_txt = zip_ref.namelist()[0]

    with zip_ref.open(nome_txt) as arquivo:
        for i in range(5):
            linha = arquivo.readline()
            print(linha)

b'00COTAHIST.2010BOVESPA 20101230                                                                                                                                                                                                                      \r\n'
b'012010010402ABCB4       010ABC BRASIL  PN  EJ  N2   R$  000000000120000000000012420000000001200000000000123200000000012350000000001234000000000123500421000000000000205500000000000253269800000000000000009999123100000010000000000000BRABCBACNPR4109\r\n'
b'012010010496ABCB4F      020ABC BRASIL  PN  EJ  N2   R$  000000000121000000000012510000000001210000000000122800000000012400000000001238000000000124000017000000000000000423000000000000520805000000000000009999123100000010000000000000BRABCBACNPR4109\r\n'
b'012010010402ABNB3       010ABNOTE      ON  ED  NM   R$  000000000189700000000019450000000001890000000000191300000000019300000000001925000000000193000240000000000000289400000000000553926000000000000000009999123100000010000000000000BRABNBACN

### 3. Definição da estrutura da COTAHIST

A base COTAHIST da B3 não é um CSV tradicional. Ela é um arquivo de largura fixa, em que cada informação ocupa uma quantidade específica de caracteres na linha. Por isso, para transformar o TXT em uma tabela, é necessário informar ao pandas os nomes das colunas e a largura de cada campo.

In [13]:
COLUNAS = [
    "tipo_registro",
    "data_pregao",
    "codigo_bdi",
    "codigo_negociacao",
    "tipo_mercado",
    "nome_empresa",
    "especificacao_papel",
    "prazo_termo",
    "moeda",
    "preco_abertura",
    "preco_maximo",
    "preco_minimo",
    "preco_medio",
    "preco_ultimo",
    "preco_melhor_oferta_compra",
    "preco_melhor_oferta_venda",
    "numero_negocios",
    "quantidade_titulos",
    "volume_financeiro",
    "preco_exercicio",
    "indicador_correcao",
    "data_vencimento",
    "fator_cotacao",
    "preco_exercicio_pontos",
    "codigo_isin",
    "numero_distribuicao"
]

LARGURAS = [
    2, 8, 2, 12, 3, 12, 10, 3, 4,
    13, 13, 13, 13, 13, 13, 13,
    5, 18, 18, 13, 1, 8, 7, 13, 12, 3
]

COLUNAS_PRECO = [
    "preco_abertura",
    "preco_maximo",
    "preco_minimo",
    "preco_medio",
    "preco_ultimo",
    "preco_melhor_oferta_compra",
    "preco_melhor_oferta_venda",
    "preco_exercicio",
    "preco_exercicio_pontos"
]

In [14]:
with zipfile.ZipFile(arquivo_teste, "r") as zip_ref:
    nome_txt = zip_ref.namelist()[0]

    with zip_ref.open(nome_txt) as arquivo:
        df_2010_bruto = pd.read_fwf(
            arquivo,
            widths=LARGURAS,
            names=COLUNAS,
            skiprows=1,
            skipfooter=1,
            encoding="latin1"
        )

df_2010_bruto.head()

,tipo_registro,data_pregao,codigo_bdi,codigo_negociacao,tipo_mercado,nome_empresa,especificacao_papel,prazo_termo,moeda,preco_abertura,...,numero_negocios,quantidade_titulos,volume_financeiro,preco_exercicio,indicador_correcao,data_vencimento,fator_cotacao,preco_exercicio_pontos,codigo_isin,numero_distribuicao
0,1,20100104,2,ABCB4,10,ABC BRASIL,PN EJ N2,NaN,R$,1200,...,421,205500,253269800,0,0,99991231,1,0,BRABCBACNPR4,109
1,1,20100104,96,ABCB4F,20,ABC BRASIL,PN EJ N2,NaN,R$,1210,...,17,423,520805,0,0,99991231,1,0,BRABCBACNPR4,109
2,1,20100104,2,ABNB3,10,ABNOTE,ON ED NM,NaN,R$,1897,...,240,289400,553926000,0,0,99991231,1,0,BRABNBACNOR4,114
3,1,20100104,96,ABNB3F,20,ABNOTE,ON ED NM,NaN,R$,1895,...,4,33,63159,0,0,99991231,1,0,BRABNBACNOR4,114
4,1,20100104,62,ABNB3T,30,ABNOTE,ON ED NM,31.0,R$,1926,...,2,4000,7707860,0,0,99991231,1,0,BRABNBACNOR4,114


### 4. Criação de uma função para estruturar a tabela de todos os anos

Após testar a leitura e o tratamento da COTAHIST com o arquivo de 2010, o próximo passo é generalizar esse processo para todos os anos da base, de 2010 a 2025.

Para isso, criamos uma função que recebe um ano como entrada, abre o arquivo ZIP correspondente, lê o TXT da COTAHIST, aplica os tratamentos necessários e retorna uma tabela padronizada. Essa função será reutilizada em um loop para processar todos os arquivos anuais e juntá-los em uma única base histórica da B3.

In [15]:
def ler_cotahist_ano(ano):
    caminho_zip = PASTA_DADOS / f"COTAHIST_A{ano}.ZIP"

    if not caminho_zip.exists():
        print(f"Arquivo não encontrado: {caminho_zip}")
        return pd.DataFrame()

    with zipfile.ZipFile(caminho_zip, "r") as zip_ref:
        nome_txt = zip_ref.namelist()[0]

        with zip_ref.open(nome_txt) as arquivo:
            df = pd.read_fwf(
                arquivo,
                widths=LARGURAS,
                names=COLUNAS,
                skiprows=1,
                skipfooter=1,
                encoding="latin1"
            )

    df["data_pregao"] = pd.to_datetime(df["data_pregao"], format="%Y%m%d")

    for coluna in COLUNAS_PRECO:
        df[coluna] = df[coluna] / 100

    df["volume_financeiro"] = df["volume_financeiro"] / 100

    df["codigo_negociacao"] = df["codigo_negociacao"].str.strip()
    df["nome_empresa"] = df["nome_empresa"].str.strip()
    df["especificacao_papel"] = df["especificacao_papel"].str.strip()

    df = df[df["tipo_mercado"] == 10]

    df = df[df["codigo_bdi"].isin([2, 96])]

    colunas_finais = [
        "data_pregao",
        "codigo_negociacao",
        "nome_empresa",
        "especificacao_papel",
        "preco_abertura",
        "preco_maximo",
        "preco_minimo",
        "preco_medio",
        "preco_ultimo",
        "numero_negocios",
        "quantidade_titulos",
        "volume_financeiro",
        "codigo_isin"
    ]

    return df[colunas_finais]

### 5. Processamento da base histórica completa da B3

Nesta etapa, aplicamos a função de leitura e tratamento para todos os arquivos COTAHIST disponíveis, de 2010 a 2025. Cada arquivo anual é processado separadamente e armazenado em uma lista. Em seguida, todas as bases anuais são unidas em uma única base histórica consolidada.

In [16]:
ANO_INICIAL = 2010
ANO_FINAL = 2025

bases_anuais = []

for ano in range(ANO_INICIAL, ANO_FINAL + 1):
    print(f"Processando ano {ano}...")

    df_ano = ler_cotahist_ano(ano)

    if not df_ano.empty:
        bases_anuais.append(df_ano)

dados_b3 = pd.concat(bases_anuais, ignore_index=True)

dados_b3 = dados_b3.sort_values(["data_pregao", "codigo_negociacao"])

dados_b3.head()

Processando ano 2010...
Processando ano 2011...
Processando ano 2012...
Processando ano 2013...
Processando ano 2014...
Processando ano 2015...
Processando ano 2016...
Processando ano 2017...
Processando ano 2018...
Processando ano 2019...
Processando ano 2020...
Processando ano 2021...
Processando ano 2022...
Processando ano 2023...
Processando ano 2024...
Processando ano 2025...


,data_pregao,codigo_negociacao,nome_empresa,especificacao_papel,preco_abertura,preco_maximo,preco_minimo,preco_medio,preco_ultimo,numero_negocios,quantidade_titulos,volume_financeiro,codigo_isin
0,2010-01-04,ABCB4,ABC BRASIL,PN EJ N2,12.00,12.42,12.00,12.32,12.35,421,205500,2532698.0,BRABCBACNPR4
1,2010-01-04,ABNB3,ABNOTE,ON ED NM,18.97,19.45,18.90,19.13,19.30,240,289400,5539260.0,BRABNBACNOR4
2,2010-01-04,ABYA3,ABYARA,ON NM,4.61,4.62,4.57,4.59,4.60,483,901100,4141436.0,BRABYAACNOR3
3,2010-01-04,ACGU3,GUARANI,ON NM,5.65,5.78,5.60,5.72,5.78,891,713800,4089290.0,BRACGUACNOR6
4,2010-01-04,AEDU11,ANHANGUERA,UNT N2,24.87,25.45,24.87,25.24,25.10,1359,452800,11432550.0,BRAEDUCDAM18


### 6. Conferência da base consolidada

Após processar todos os arquivos anuais da COTAHIST, realizamos uma conferência geral da base consolidada. O objetivo é verificar o tamanho da base, o período coberto, a quantidade de ativos únicos e a consistência inicial das principais colunas.

In [17]:
print(f"Base completa: {dados_b3.shape[0]} linhas e {dados_b3.shape[1]} colunas")
print(f"Data inicial: {dados_b3['data_pregao'].min()}")
print(f"Data final: {dados_b3['data_pregao'].max()}")
print(f"Quantidade de ativos únicos: {dados_b3['codigo_negociacao'].nunique()}")

dados_b3[[
    "data_pregao",
    "codigo_negociacao",
    "nome_empresa",
    "especificacao_papel",
    "preco_ultimo",
    "numero_negocios",
    "quantidade_titulos",
    "volume_financeiro"
]].head(10)

Base completa: 1573367 linhas e 13 colunas
Data inicial: 2010-01-04 00:00:00
Data final: 2025-12-30 00:00:00
Quantidade de ativos únicos: 2051


,data_pregao,codigo_negociacao,nome_empresa,especificacao_papel,preco_ultimo,numero_negocios,quantidade_titulos,volume_financeiro
0,2010-01-04,ABCB4,ABC BRASIL,PN EJ N2,12.35,421,205500,2532698.0
1,2010-01-04,ABNB3,ABNOTE,ON ED NM,19.30,240,289400,5539260.0
2,2010-01-04,ABYA3,ABYARA,ON NM,4.60,483,901100,4141436.0
3,2010-01-04,ACGU3,GUARANI,ON NM,5.78,891,713800,4089290.0
4,2010-01-04,AEDU11,ANHANGUERA,UNT N2,25.10,1359,452800,11432550.0
5,2010-01-04,AELP3,AES ELPA,ON,42.00,2,700,29400.0
6,2010-01-04,AGEN11,AGRENCO,DR3,2.79,1536,4546300,12736907.0
7,2010-01-04,AGIN3,AGRA INCORP,ON NM,5.00,4916,2496000,12380446.0
8,2010-01-04,AGRO3,BRASILAGRO,ON NM,10.20,2,6000,61450.0
9,2010-01-04,ALLL11,ALL AMER LAT,UNT N2,17.05,4491,2341500,39318963.0


### 7. Salvamento da base consolidada e criação das matrizes

Após conferir a base histórica consolidada da B3, salvamos a tabela completa em formato Parquet, que é mais eficiente para armazenar e carregar bases grandes no Python.

Em seguida, transformamos a base em três matrizes principais: preços, volume financeiro e número de negócios. Nessas matrizes, cada linha representa uma data de pregão, cada coluna representa um ativo e cada célula contém o valor correspondente daquele ativo naquela data.

In [18]:
PASTA_DADOS_TRATADOS.mkdir(parents=True, exist_ok=True)

dados_b3.to_parquet(
    PASTA_DADOS_TRATADOS / "cotahist_b3_2010_2025.parquet",
    index=False
)

precos = dados_b3.pivot_table(
    index="data_pregao",
    columns="codigo_negociacao",
    values="preco_ultimo",
    aggfunc="last"
)

volumes_financeiros = dados_b3.pivot_table(
    index="data_pregao",
    columns="codigo_negociacao",
    values="volume_financeiro",
    aggfunc="sum"
)

negocios = dados_b3.pivot_table(
    index="data_pregao",
    columns="codigo_negociacao",
    values="numero_negocios",
    aggfunc="sum"
)

precos.to_csv(PASTA_DADOS_TRATADOS / "precos_b3_2010_2025.csv")
volumes_financeiros.to_csv(PASTA_DADOS_TRATADOS / "volumes_financeiros_b3_2010_2025.csv")
negocios.to_csv(PASTA_DADOS_TRATADOS / "negocios_b3_2010_2025.csv")

print("Arquivos salvos com sucesso.")
print(f"Matriz de preços: {precos.shape[0]} datas e {precos.shape[1]} ativos")
print(f"Matriz de volume financeiro: {volumes_financeiros.shape[0]} datas e {volumes_financeiros.shape[1]} ativos")
print(f"Matriz de negócios: {negocios.shape[0]} datas e {negocios.shape[1]} ativos")

Arquivos salvos com sucesso.
Matriz de preços: 3967 datas e 2051 ativos
Matriz de volume financeiro: 3967 datas e 2051 ativos
Matriz de negócios: 3967 datas e 2051 ativos
